In [ ]:
import time
from datetime import datetime
import pandas as pd
import re
from apscheduler.schedulers.background import BackgroundScheduler
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=chrome_options)

URL = "https://www.sothailand.org/genws/dashboard"
data_history = []

def save_data():
    if data_history:
        df = pd.DataFrame(data_history)
        df.to_csv("scraped_power_data.csv", index=False)
        return df
    return None

def scrape_and_store():
    driver.get(URL)
    time.sleep(3)
    
    wait = WebDriverWait(driver, 5)
    
    date_element = driver.find_elements(By.XPATH, "//div[contains(@class, 'date')]//span[1]")
    date_str = date_element[0].text if date_element else datetime.now().strftime("%d-%m-%Y")
    
    time_element = driver.find_elements(By.XPATH, "//div[contains(@class, 'time')]//span")
    time_str = time_element[0].text if time_element else datetime.now().strftime("%H:%M")
    
    power_element = driver.find_elements(By.XPATH, "//span[contains(@style,'font-size')]")
    if power_element:
        current_value = power_element[0].text.split()[0].replace(',', '')
    else:
        elements = driver.find_elements(By.XPATH, "//div[contains(text(), 'MW')]")
        if elements:
            numbers = re.findall(r'[\d,]+\.?\d*', elements[0].text)
            current_value = numbers[0].replace(',', '') if numbers else "0"
        else:
            current_value = "0"
    
    temp_element = driver.find_elements(By.XPATH, "//span[contains(text(), '°C')]")
    temp = temp_element[0].text.replace('°C', '').strip() if temp_element else "0"
    
    now = datetime.now()
    power_value = float(current_value) if current_value.replace('.', '', 1).isdigit() else 0.0
    temp_value = float(temp) if temp.replace('.', '', 1).isdigit() else 0.0
    
    data = {
        'scrape_time': now,
        'display_date': date_str,
        'display_time': time_str,
        'current_value_MW': power_value,
        'temperature_C': temp_value,
    }
    
    data_history.append(data)
    
    if len(data_history) % 5 == 0:
        save_data()

sched = BackgroundScheduler()
sched.add_job(scrape_and_store, 'interval', minutes=1)
sched.start()

scrape_and_store()

while True:
    time.sleep(60)
    if len(data_history) % 10 == 0:
        save_data()